In [1]:
import sys, os

In [8]:
pip install langchain_community

  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.5 MB/s  0:00:00 9.8 MB/s eta 0:00:01
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [langchain_community]m4/5 [langchain_community]
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import numpy as np 
import re, json 
import time
import sqlite3
import requests
import io
from datetime import datetime

from urllib.parse import urljoin

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as ExpectedConditions
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

from IPython.display import Image
from IPython.display import display, HTML

#from langchain_community.llms import Ollama


In [9]:
import typing_extensions

from langchain_community.llms import Ollama

/var/folders/cc/fgdvcyz155z2b_s9qh66r2pw0000gn/T/ipykernel_43443/2176515609.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Ollama


In [10]:
from langchain_ollama import OllamaLLM

In [11]:
# options = Options()
# options.binary_location = '/usr/bin/google-chrome'
# options.add_argument('--headless')
# options.add_argument('--disable-gpu')

# CHROME_PATH = '/usr/lib/chromium-browser/chromedriver'
# log_name = '/tmp/ghostdriver.log'
# browser  = webdriver.Chrome(executable_path=CHROME_PATH, chrome_options=options, service_log_path=log_name)
# browser2  = webdriver.Chrome(executable_path=CHROME_PATH, chrome_options=options, service_log_path=log_name)

# CHROMEDRIVER_PATH = "/usr/lib/chromium-browser/chromedriver"
# LOG_PATH =  '/tmp/ghostdriver.log'
# # Set up Chrome options (optional, e.g., headless mode)
# chrome_options = Options()
# chrome_options.add_argument("--headless")
# chrome_options.add_argument("--no-sandbox")
# chrome_options.add_argument("--disable-gpu")
# # Initialize the ChromeDriver
# service = Service(
#     executable_path=CHROMEDRIVER_PATH,
#     log_path=LOG_PATH
# )
# browser = webdriver.Chrome(service=service, options=chrome_options)
# browser2 = webdriver.Chrome(service=service, options=chrome_options)


LOG_PATH =  '/tmp/ghostdriver.log'
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-gpu')
options.add_argument('--disable-dev-shm-usage')
browser = webdriver.Chrome(service=Service(ChromeDriverManager().install(), log_path=LOG_PATH), options=options)
browser2 = webdriver.Chrome(service=Service(ChromeDriverManager().install(), log_path=LOG_PATH), options=options)




In [12]:
#!pip install -U langchain-ollama 

#from langchain_ollama import OllamaLLM

#llm = Ollama(base_url="http://cam0.ddns.net:11434/", model="llama3.1")
llm = OllamaLLM(base_url="http://localhost:11434/", model="llama3.1")
#llm = OllamaLLM(model="llama3.1")

In [16]:
llm = OllamaLLM(base_url="http://cam0.ddns.net:11435/", model="llama3.1")

NameError: name 'OllamaLLM' is not defined

In [12]:
from ollama import Client
client = Client(host='http://localhost:11434', timeout=300) # Set to 5 minutes


In [13]:
llm.invoke('how many capitals are in France?')

ConnectError: [Errno 61] Connection refused

In [15]:
# EPD 

url = 'https://www.epd.gov.hk/epd/english/news_events/notices/notices.html'
browser.get (url)
soup = BeautifulSoup(browser.page_source, 'html.parser')
batch_time = datetime.now() 

urlbase = url[0:url.rfind('/')+1]
df = pd.DataFrame()

for d in soup.select ('div[class~=node__content]'):
    for l in d.select ('li'):
        for a in l.select ('a'):
            url2 = urlbase + a.attrs['href']
            browser2.get (url2)
            soup2 = BeautifulSoup(browser2.page_source, 'html.parser')
            for d2 in soup2.select ('div[class~=node__content]'):
                for t2 in d2.select ('table'):
                    # data = pd.read_html(str(t2))
                    data = pd.read_html(io.StringIO(str(t2)))
                    data[0]['url']=url2
                    df = df._append(data[0])

df['batch_time'] = batch_time
df['department'] = 'Environmental Protection Department'
df.to_json('/Users/rowena/data/epd_raw.json', orient = 'records', compression = 'infer', index = 'false')

df2 = pd.read_json('/Users/rowena/data/epd_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()

epd_date_pattern = r"\d{1,2} (January|February|March|April|May|June|July|August|September|October|November|December|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)( |\xa0)\d{2,4}"

def epd_get_period (row): 
    if 'Awarded Period' in df.columns: 
        return row['Awarded Period']
    if 'Contract Period' in df.columns: 
        return row['Contract Period']
    if row['Awarded Quantity'] is not None: 
        return row['Awarded Quantity']
    else: 
        return '' 


def get_cesd2_date (row): 
    if row['Award Date'] is not None :
        return datetime.strptime(row['Award Date'], '%d/%m/%Y').date().strftime('%d %B %Y')
    else:
        return ''
    
def epd_get_date (row): 
    if row['Estimated Awarded Sum / Contract Award Date'] is not None: 
        x = row['Estimated Awarded Sum / Contract Award Date']
        y = re.search (epd_date_pattern,x, re.IGNORECASE)
        if y is not None:
            return y.group().strip()
        else:
            return ''

def epd_get_sum (row): 
    if row['Estimated Awarded Sum / Contract Award Date'] is not None: 
        x = row['Estimated Awarded Sum / Contract Award Date']
        y = re.search (epd_date_pattern,x, re.IGNORECASE)
        if y is not None:
            return x.lower().replace(y.group(),'').strip()
        else:
            return ''
    if row['Estimated Awarded Sum'] is not None: 
        return 'HK$ ' + str(row['Estimated Awarded Sum']).strip()    
    elif row['Estimated Awarded Sum (HK$)'] is not None: 
        return 'HK$ ' + str(row['Estimated Awarded Sum (HK$)']).strip()
    else: 
        return '' 


def epd_get_awardee (row): 
    prompt = "Valid examples of company name include: \nAECOM Asia Company Limited\nATAL Engineering Limited\nBased on the example provided and other common company name format, try to extract company name from following. No need to show any explanation. :\n"
    return llm.invoke(prompt + str(row['Contractor(s) and Address(es)']))

#row['Contractor(s) and Address(es)']

# ref, department, office, description, awardee, award_date, sum,  period, start, end url, batch_time
df3[['ref']]=df2[['Tender Reference']]
df3['department'] = df2['department']
df3[['description']]=df2[['Subject']]
df3['awardee']=df2.apply(epd_get_awardee, axis=1) 
df3['award_date'] =df2.apply(epd_get_date, axis=1) 
df3['sum'] =df2.apply(epd_get_sum, axis=1) 
df3['period'] = df2.apply(epd_get_period, axis=1) 
df3[['url']]=df2[['url']]
df3['batch_time']=df2['batch_time']

 
df3.to_json('/Users/rowena/data/epd_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/epd_out.csv', index=False)  

In [ ]:
df.columns

In [ ]:
def epd_get_awardee (row): 
    prompt = "Valid examples of company name include: \nAECOM Asia Company Limited\nATAL Engineering Limited\nBased on the example provided and other common company name format, try to extract company name from following. No need to show any explanation. :\n"
    return llm.invoke(prompt + str(row))

df3['awardee']=df2['Contractor(s) and Address(es)'].map(epd_get_awardee)

In [ ]:
def epd_get_period (row): 
    if row['Awarded Period'] is not None: 
        return row['Awarded Period']
    # elif row['Contract Period'] is not None: 
    #     return row['Contract Period']
    elif row['Awarded Quantity'] is not None: 
        return row['Awarded Quantity']
    else: 
        return '' 


df3['award_date'] = df2.apply(epd_get_date, axis=1) 
df3['sum'] = df2.apply(epd_get_sum, axis=1) 
df3['period'] = df2.apply(epd_get_period, axis=1) 
df3[['url']]=df2[['url']]
df3['batch_time']=df2['batch_time']

 
df3.to_json('/Users/rowena/data/epd_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/epd_out.csv', index=False)  

In [ ]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.1")
response = llm.invoke("Explain LangChain in one sentence.")
print(response)


In [17]:
# DSD
# pending to add 2nd page 

url = 'https://www.dsd.gov.hk/EN/Our_Projects/Contracts_Consultancies_Awarded/index.html'
browser.get (url)
browser.implicitly_wait(10)
soup = BeautifulSoup(browser.page_source, 'html.parser')
batch_time = datetime.now() 

df = pd.DataFrame()
data = []

for a in browser.find_elements("xpath","//table[contains(@id, 'resultTable')]//a"):
    url2 = a.get_attribute('href') 
    browser2.get (url2)
    soup2 = BeautifulSoup(browser2.page_source, 'html.parser')
    d = soup2.select('div[class~=detail]')[0]
    item = {}
    item ['ref'] = d.select('div[class~=contract_no]')[0].text
    item ['description'] = d.select('div[class~=contract_no]')[0].next_sibling.next_sibling.text
    item ['url'] = url2
    for r in d.select('div[class~=content]')[0].select('div[class~=row]'):
        subject = r.select('div')[0].text
        value = r.select('div')[1].text
        item[subject] = value
    data.append(item)
        
df = pd.DataFrame(data)
df['batch_time'] = batch_time
df['department'] = 'Drainage Services Department'
df.to_json('/Users/rowena/data/dsd_raw.json', orient = 'records', compression = 'infer', index = 'false')

df2 = pd.read_json('/Users/rowena/data/dsd_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()

def get_dsd_awardee (row): 
    if row['Contractor'] is not None: 
        return row['Contractor']
    if row['Consultant'] is not None: 
        return row['Consultant']
    else: 
        return '' 

def get_dsd_sum (row): 
    if row['Contract Sum'] is not None: 
        return row['Contract Sum']
    if row['Awarded Consultancy Fee'] is not None: 
        return row['Awarded Consultancy Fee']
    else: 
        return '' 
    
# ref, department, office, description, awardee, award_date, sum,  period, start, end url, batch_time
df3['ref']=df2['ref']
df3['department'] = df2['department']
df3['description']=df2['description']
df3['office']=df2['Project Office']
df3['awardee'] = df2.apply(get_dsd_awardee, axis=1) 
df3['award_date']=df2['Date of Award']
df3['sum'] = df2.apply(get_dsd_sum, axis=1) 
df3['start']=df2['Date of Commencement']
df3['end']=df2['Anticipated Date of Completion']
df3['url']=df2['url']
df3['batch_time'] = df2['batch_time']

df3.to_json('/Users/rowena/data/dsd_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/dsd_out.csv', index=False)  



In [19]:
# CEDD
# pending to fix description for extended html content

url = 'https://www.cedd.gov.hk/eng/tender-notices/contracts/contracts-awarded/index.html'
browser.get (url)
browser.implicitly_wait(10)
soup = BeautifulSoup(browser.page_source, 'html.parser')
batch_time = datetime.now() 

df = pd.DataFrame()
data = []

for tr in browser.find_elements("xpath","//div[contains(@id, 'content')]//tbody//tr"):
    url2 = tr.find_element("xpath",".//a").get_attribute('href')
    item ['award_date'] = tr.find_elements("xpath","td")[2].text
    browser2.get (url2)
    soup2 = BeautifulSoup(browser2.page_source, 'html.parser')
    d = soup2.select('div[id~=content]')[0]
    item = {}
    item ['url'] = url2
    for r in d.select('h3'):
        subject = r.text.replace(' :','')
        value = r.next_sibling
        item[subject] = value
    data.append(item)

df = pd.DataFrame(data)
df['batch_time'] = batch_time
df['department'] = 'Civil Engineering and Development Department'

df.to_json('/Users/rowena/data/cesd_raw.json', orient = 'records', compression = 'infer', index = 'false')
df2 = pd.read_json('/Users/rowena/data/cesd_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()

def get_cesd_desc (row): 
    return BeautifulSoup(str(row['Description']), 'html.parser').text

def get_cesd_date (row): 
    if row['award_date'] is not None:
        return datetime.strptime(row['award_date'], '%d/%m/%Y').date().strftime('%d %B %Y')
    else:
        return ''

# ref, department, office, description, awardee, award_date, sum,  period, start, end url, batch_time
df3['ref']=df2['Tender Reference']
df3['department'] = df2['department']
df3['description']=df2.apply(get_cesd_desc, axis=1) 
df3['awardee'] = df2['Contractor']
df3['award_date']=df2.apply(get_cesd_date, axis=1) 
df3['sum'] = df2['Awarded Sum (million)'] + ' (m)'
df3['url']=df2['url']
df3['batch_time'] = df2['batch_time']

df3.to_json('/Users/rowena/data/cesd_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/cesd_out.csv', index=False)  

In [21]:
# CEDD consultancies
# pending to fix description for extended html content

url = 'https://www.cedd.gov.hk/eng/tender-notices/consultancy-agreements/consultancies-awarded/index.html'
browser.get (url)
browser.implicitly_wait(10)
soup = BeautifulSoup(browser.page_source, 'html.parser')
batch_time = datetime.now() 

data = pd.read_html(io.StringIO(str(soup.select("div[id~=content]")[0])))
df = data[0]
df['department'] = "Civil Engineering and Development Department"
df['url'] = url
df['batch_time'] = batch_time

df.to_json('/Users/rowena/data/cesd2_raw.json', orient = 'records', compression = 'infer', index = 'false')
df2 = pd.read_json('/Users/rowena/data/cesd2_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()

def get_cesd2_date (row): 
    if row['Award Date'] is not None :
        return datetime.strptime(row['Award Date'], '%d/%m/%Y').date().strftime('%d %B %Y')
    else:
        return ''

# ref, department, office, description, awardee, award_date, sum,  period, start, end url, batch_time
df3['ref']=df2['Agreement Number']
df3['department'] = df2['department']
df3['description']=df2['Agreement Title']
df3['awardee'] = df2['Name of Successful Consultant(s)']
df3['award_date']=df2.apply(get_cesd2_date, axis=1) 
df3['sum'] = df2['Fee']
df3['start'] = df2['Commencement Date']
df3['end'] = df2['Estimated Completion Date']
df3['url']=df2['url']
df3['batch_time'] = df2['batch_time']

df3.to_json('/Users/rowena/data/cesd2_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/cesd2_out.csv', index=False)  


In [23]:
# EMSD

url = 'https://www.emsd.gov.hk/en/tenders_contracts_and_consultancies/tender_notices/award_of_tender/index.html'
browser.get (url)
browser.implicitly_wait(10)
soup = BeautifulSoup(browser.page_source, 'html.parser')
batch_time = datetime.now() 

df = pd.DataFrame()
data = []
for t in soup.select ("div[id~=tendernotice_awardcontent]")[0].select("table"):
    item = {}
    for r in t.select("tr"):
        subject = r.select("th")[0].text
        value = r.select("td")[0].text.replace('\xa0',' ')
        item[subject] = value
    data.append(item)

df = pd.DataFrame(data)
df['url'] = url
df['batch_time'] = batch_time
df['department'] = 'Electrical and Mechanical Services Department'

df.to_json('/Users/rowena/data/emsd_raw.json', orient = 'records', compression = 'infer', index = 'false')

df2 = pd.read_json('/Users/rowena/data/emsd_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()

# ref, department, office, description, awardee, award_date, sum,  period, start, end url, batch_time
df3['ref']=df2['Contract Number']
df3['department'] = df2['department']
df3['description']=df2['Contract Title'] + ' / ' + df2['Nature of Works'] 
df3['awardee'] = df2['Contractor(s)']
df3['award_date']=df2['Date of award']

def get_emsd_sum (row): 
    if row['Original Contract Sum'] is not None: 
        return row['Original Contract Sum']
    if row['Contract Value'] is not None: 
        return row['Contract Value']
    else: 
        return '' 

df3['sum'] = df2.apply(get_emsd_sum, axis=1)  
df3['start'] = df2['Commencement Date']
df3['end'] = df2 ['Original Completion Date']
df3['url']=df2['url']
df3['batch_time'] = df2['batch_time']
df3.to_json('/Users/rowena/data/emsd_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/emsd_out.csv', index=False)   



In [25]:
# EMSD consultancies

url = 'https://www.emsd.gov.hk/en/tenders_contracts_and_consultancies/tender_notices/award_of_consultancies/index.html'
browser.get (url)
browser.implicitly_wait(10)
soup = BeautifulSoup(browser.page_source, 'html.parser')
batch_time = datetime.now() 

df = pd.DataFrame()
data = []
for t in soup.select ("table[class~=tab_list]"):
    item = {}
    for r in t.select("tr"):
        subject = r.select("th")[0].text
        value = r.select("td")[0].text.replace('\xa0',' ').replace('\n','')
        item[subject] = value
    data.append(item)

df = pd.DataFrame(data)

df['url'] = url
df['batch_time'] = batch_time
df['department'] = 'Electrical and Mechanical Services Department'

df.to_json('/Users/rowena/data/emsd2_raw.json', orient = 'records', compression = 'infer', index = 'false')

df2 = pd.read_json('/Users/rowena/data/emsd2_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()

# ref, department, office, description, awardee, award_date, sum,  period, start, end url, batch_time
df3['ref']=df2['Agreement No.']
df3['department'] = df2['department']
df3['description']=df2['Consultancy Title']  
df3['awardee'] = df2['Consultancy Name']
df3['award_date']=df2['Award Date']
df3['sum'] = df2['Consultancy Cost']
df3['end'] = df2 ['Estimated Completion Date']
df3['url']=df2['url']
df3['batch_time'] = df2['batch_time']
df3.to_json('/Users/rowena/data/emsd2_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/emsd2_out.csv', index=False)   



In [27]:
# HKAA

url = 'https://www.hongkongairport.com/en/airport-authority/tender-notices/notice-of-contract-award.page'
browser.get (url)
browser.implicitly_wait(10)
soup = BeautifulSoup(browser.page_source, 'html.parser')
batch_time = datetime.now() 

df = pd.DataFrame()

data = []
for d in soup.select ("div[class~=contractData]"):
    item = {}
    item['ref'] = d.text
    item['description'] = d.find_next_sibling().find("a").text
    url2 = urljoin (url, d.find_next_sibling().find("a").get('href'))
    item['url'] =  url2
    item ['award_date'] = d.find_next_sibling().find_next_sibling().text
    url2 = urljoin (url, d.find_next_sibling().find("a").get('href'))
    browser2.get (url2)
    soup2 = BeautifulSoup(browser2.page_source, 'html.parser')
    item['description'] = soup2.select ("div[class~=tenderDetail]")[0].text.replace('\nNotice of Contract Award\n','')
    data.append(item)

df = pd.DataFrame(data)

df['batch_time'] = batch_time
df['department'] = 'Hong Kong Airport Authority'

df.to_json('/Users/rowena/data/hkaa_raw.json', orient = 'records', compression = 'infer', index = 'false')

df2 = pd.read_json('/Users/rowena/data/hkaa_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()

# ref, department, office, description, awardee, award_date, sum,  period, start, end url, batch_time

def get_hkaa_sum (row): 
    x =  re.search (r'HK\$[\d,.]+( )?(million)?' ,row['description'])
    if x:
        return x[0]
    else: 
        return ''

def get_hkaa_date (row): 
    if row['award_date'] is not None :
        return datetime.strptime(row['award_date'], '%Y-%m-%d').date().strftime('%d %B %Y')
    else:
        return ''

def get_hkaa_awardee (row): 
    prompt = "Valid examples of company name include: \nAECOM Asia Company Limited\nATAL Engineering Limited\nBased on the example provided and other common company name format, try to identify company who got the contract award from following. No need to show any explanation. :\n"
    return llm.invoke(prompt + row['description'])

df3['ref']=df2['ref']
df3['description']=df2['description'] 
df3['url']=df2['url']
df3['awardee'] = df2.apply(get_hkaa_awardee, axis=1)  
df3['award_date']=df2.apply(get_hkaa_date, axis=1)  
df3['sum'] = df2.apply(get_hkaa_sum, axis=1)  
df3['batch_time'] = df2['batch_time']
df3['department'] = df2['department']

df3.to_json('/Users/rowena/data/hkaa_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/hkaa_out.csv', index=False)   



In [28]:
# HYD

url = 'https://www.hyd.gov.hk/en/tender_notices/contracts/awarded/index.html'
browser.get (url)
browser.implicitly_wait(10)
soup = BeautifulSoup(browser.page_source, 'html.parser')
batch_time = datetime.now() 
data = pd.read_html(io.StringIO(str(soup.select("div[id~=content]")[0])))
df = data[0]
df['batch_time'] = batch_time
df['department'] = 'Highways Department'
df['url']= url
df.to_json('/Users/rowena/data/hyd_raw.json', orient = 'records', compression = 'infer', index = 'false')

df2 = pd.read_json('/Users/rowena/data/hyd_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()

def get_hyd_date (row): 
    if row['Award Date (dd/mm/yyyy)'] is not None :
        return datetime.strptime(row['Award Date (dd/mm/yyyy)'], '%d/%m/%Y').date().strftime('%d %B %Y')
    else:
        return ''
    
df3['ref']=df2['Contract No.']
df3['description']=df2['Contract Title'] 
df3['url']=df2['url']
df3['award_date']=df2.apply(get_hyd_date, axis=1)  
df3['awardee']=df2['Contractor Name']
df3['sum']='HK$ ' + df2['Contract Sum ($M)'].astype(str) + '(m)'
df3['batch_time'] = df2['batch_time']
df3['department'] = df2['department']

df3.to_json('/Users/rowena/data/hyd_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/hyd_out.csv', index=False)   


In [31]:
# HYD consultancies

url = 'https://www.hyd.gov.hk/en/tender_notices/consultancies/awarded/index.html'
browser.get (url)
browser.implicitly_wait(10)
soup = BeautifulSoup(browser.page_source, 'html.parser')
batch_time = datetime.now() 
data = pd.read_html(io.StringIO(str(soup.select("div[id~=content]")[0])))
df = data[0]
df['batch_time'] = batch_time
df['department'] = 'Highways Department'
df['url']= url
df.to_json('/Users/rowena/data/hyd2_raw.json', orient = 'records', compression = 'infer', index = 'false')

df2 = pd.read_json('/Users/rowena/data/hyd2_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()

def get_hyd2_date (row): 
    if row['Award Date  (dd/mm/yyyy)'] is not None :
        return datetime.strptime(row['Award Date  (dd/mm/yyyy)'], '%d/%m/%Y').date().strftime('%d %B %Y')
    else:
        return ''
    
df3['ref']=df2['Agreement  No.']
df3['description']=df2['Agreement Title'] 
df3['url']=df2['url']
df3['award_date']=df2.apply(get_hyd2_date, axis=1)  
df3['awardee']=df2['Contractor Name']
df3['sum']='HK$ ' + df2['Consultancy Fee ($M)'].astype(str) + '(m)'
df3['batch_time'] = df2['batch_time']
df3['department'] = df2['department']

df3.to_json('/Users/rowena/data/hyd2_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/hyd2_out.csv', index=False)   


In [33]:
# WSD

url = 'https://www.wsd.gov.hk/en/tenders-contracts-and-consultancies/contracts/active-wsd-capital-works-contracts/index.html'
browser.get (url)
browser.implicitly_wait(120)
batch_time = datetime.now() 
time.sleep(10)
# browser.implicitly_wait(180)
soup = BeautifulSoup(browser.page_source, 'html.parser')
data = pd.read_html(io.StringIO(str(soup.select("div[class~=tableResponsive]")[0])))
df = data[0]

df['batch_time'] = batch_time
df['department'] = 'Water Supplies Department'
df['url']= url
df.to_json('/Users/rowena/data/wsd_raw.json', orient = 'records', compression = 'infer', index = 'false')



df2 = pd.read_json('/Users/rowena/data/wsd_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()

def get_wsd_date (row): 
    if row['Awarded Date'] is not None :
        return datetime.strptime(row['Awarded Date'], '%d %b %Y').date().strftime('%d %B %Y')
    else:
        return ''
    
df3['ref']=df2['PWP No.'] + ' ' + df2['Contract No.']
df3['description']=df2['Contract Title'] 
df3['url']=df2['url']
df3['awardee'] = df2['Contractor']
df3['award_date']=df2.apply(get_wsd_date, axis=1)  
df3['batch_time'] = df2['batch_time']
df3['department'] = df2['department']
df3['sum'] = 'HK$ ' + df2 ['Contract Sum* (HK$ M)'].astype(str) + ' (m)'
df3['office'] = df2['Responsible Division']

df3.to_json('/Users/rowena/data/wsd_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/wsd_out.csv', index=False)   



In [34]:
# WSD consultancies

url = 'https://www.wsd.gov.hk/en/tenders-contracts-and-consultancies/consultancies/award-consultancies/index.html'
browser.get (url)
browser.implicitly_wait(120)
time.sleep(10)
#soup = BeautifulSoup(browser.page_source, 'html.parser')
#soup = BeautifulSoup(browser.page_source, 'lxml')
soup = BeautifulSoup(browser.page_source, features='html')

batch_time = datetime.now() 

df = pd.DataFrame()
for d in soup.select("div[class~=tableResponsive]>table"): 
    data = pd.read_html(io.StringIO(str(d)))
    data[0].rename(columns=lambda x: x.replace(chr(160),chr(32)) , inplace=True)
    df = df._append(data[0])

df['batch_time'] = batch_time
df['department'] = 'Water Supplies Department'
df['url']= url
df.to_json('/Users/rowena/data/wsd2_raw.json', orient = 'records', compression = 'infer', index = 'false')

df2 = pd.read_json('/Users/rowena/data/wsd2_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()
# ref, department, office, description, awardee, award_date, sum,  period, start, end url, batch_time
def get_wsd2_date (row): 
    if row['Commencement Date'] is not None :
        return datetime.strptime(row['Commencement Date'], '%d/%m/%Y').date().strftime('%d %B %Y')
    else:
        return ''
    
df3['ref']=df2['Agreement No. (1)']
df3['department'] = df2['department']
df3['description']=df2['Agreement Title'] 
df3['awardee'] = df2['Name of Successful Consultant']
df3['award_date']=df2.apply(get_wsd2_date, axis=1)  
df3['sum'] = df2['Consultancy Cost (2)']
df3['start']=df2['Commencement Date']
df3['end']=df2['Estimated Completion Date']
df3['url']=df2['url']
df3['batch_time'] = df2['batch_time']

df3.to_json('/Users/rowena/data/wsd2_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/wsd2_out.csv', index=False)   


In [37]:
# TD

url = 'https://www.td.gov.hk/en/tender_notices/award_of_contracts_and_consultancies/works_contract/index.html'
browser.get (url)
browser.implicitly_wait(10)
soup = BeautifulSoup(browser.page_source, 'html.parser')
batch_time = datetime.now() 

df = pd.DataFrame()
t = soup.select("table[class~=content_table1]")
data = pd.read_html(io.StringIO(str(t)))
df = data[0]
      
df['batch_time'] = batch_time
df['department'] = 'Transport Department'
df['url']= url
df.to_json('/Users/rowena/data/td_raw.json', orient = 'records', compression = 'infer', index = 'false')

df2 = pd.read_json('/Users/rowena/data/td_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()
# ref, department, office, description, awardee, award_date, sum,  period, start, end url, batch_time

def get_td_date (row): 
    return datetime.strptime(row['Date of Award'], '%d.%m.%Y').date().strftime('%d %B %Y')
        
df3['ref']=df2['Agreement No. and Title']
df3['department'] = df2['department']
df3['awardee'] = df2['Name of Contractor']
# df3['award_date']=df2['Date of Award']
df3['award_date']= df2.apply(get_td_date, axis=1)  
df3['sum'] = 'HKD ' + df2['Contract Value ($M)'].astype(str) + ' (m)'
df3['period']=df2['Contract Duration (Month)']
df3['url']=df2['url']
df3['batch_time'] = df2['batch_time']

df3.to_json('/Users/rowena/data/td_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/td_out.csv', index=False)   


In [39]:
# TD consultancies

url = 'https://www.td.gov.hk/en/tender_notices/award_of_contracts_and_consultancies/non_works_contract/index.html'
browser.get (url)
browser.implicitly_wait(10)
soup = BeautifulSoup(browser.page_source, 'html.parser')
batch_time = datetime.now() 

df = pd.DataFrame()
t = soup.select("table[class~=content_table1]")
data = pd.read_html(io.StringIO(str(t)))
df = data[0]

df['batch_time'] = batch_time
df['department'] = 'Transport Department'
df['url']= url
df.to_json('/Users/rowena/data/td2_raw.json', orient = 'records', compression = 'infer', index = 'false')

df2 = pd.read_json('/Users/rowena/data/td2_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()
# ref, department, office, description, awardee, award_date, sum,  period, start, end url, batch_time

def get_td2_date (row): 
    return datetime.strptime(row['Date of Award'], '%d.%m.%Y').date().strftime('%d %B %Y')

df3['ref']=df2['Agreement No. and Title']
df3['department'] = df2['department']
df3['awardee'] = df2['Name of Contractor']
df3['award_date']= df2.apply(get_td2_date, axis=1)  
df3['sum'] = 'HKD ' + df2['Contract Value ($M)'].astype(str) + ' (m)'
df3['period']=df2['Contract Duration (Month)']
df3['url']=df2['url']
df3['batch_time'] = df2['batch_time']

df3.to_json('/Users/rowena/data/td2_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/td2_out.csv', index=False)   


In [41]:
# GLD master

url = 'https://pcms2.gld.gov.hk/iprod/#/scn00101'
browser.get (url)
browser.implicitly_wait(60)
soup = BeautifulSoup(browser.page_source, 'html.parser')
batch_time = datetime.now() 

df = pd.DataFrame()

for t in browser.find_elements("xpath","//table[contains(@class, 'tenderNotice')]"):
    soup2 = BeautifulSoup(t.get_attribute('outerHTML'), 'html.parser')
    department = ''
    date=[]
    sum=[]
    for tr in soup2.find ('table').find_all('tr'):
        if str(tr).find ('deptHeader') > 0:    
            if department != '' : 
                x = x + '</table>'
                data = pd.read_html (io.StringIO(x))
                data[0]['department'] = department
                df = df._append(data[0])
            x = '<table>'
            department = tr.select('span[class=deptName]')[0].text
        if str(tr).find ('columnHeader') > 0:
            x = x + str(tr)
        if str(tr).find ('monthComment') == -1 and str(tr).find ('deptHeader') == -1 and str(tr).find ('columnHeader') == -1:
            x = x + str(tr)

    data = pd.read_html (io.StringIO(x))
    data[0]['department'] = department
    df = df._append(data[0])

df['batch_time'] = batch_time
df['url']= url
df.to_json('/Users/rowena/data/gld_raw.json', orient = 'records', compression = 'infer', index = 'false')

df2 = pd.read_json('/Users/rowena/data/gld_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()
# ref, department, office, description, awardee, award_date, sum,  period, start, end url, batch_time

def get_gld_date (row): 
    x = re.search(r'\d{1,2} (Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) \d{4}$', row['Amount / Contract Award Date'])
    d = re.search(r'^\d{1,2}',x[0])[0]
    if ((re.search(r'(Jan|Mar|May|Jul|Aug|Oct|Dec)',x[0])) and int(d)>31) or  ((re.search(r'(Apr|Jun|Sep|Nov)',x[0])) and int(d)>30) or  ((re.search(r'(Feb)',x[0])) and int(d)>28):
        x2 = str(x[0])[-(len(x[0])-1):]
    else:
        x2 = x[0]
    return datetime.strptime(x2, '%d %b %Y').date().strftime('%d %B %Y')

def get_gld_sum (row): 
    x = re.search(r'\d{1,2} (Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) \d{4}$', row['Amount / Contract Award Date'])
    d = re.search(r'^\d{1,2}',x[0])[0]
    if ((re.search(r'(Jan|Mar|May|Jul|Aug|Oct|Dec)',x[0])) and int(d)>31) or  ((re.search(r'(Apr|Jun|Sep|Nov)',x[0])) and int(d)>30) or  ((re.search(r'(Feb)',x[0])) and int(d)>28):
        x2 = str(x[0])[-(len(x[0])-1):]
    else:
        x2 = x[0]
    return row['Amount / Contract Award Date'][:len(row['Amount / Contract Award Date'])-len(x2)]


def get_gld_awardee (row): 
    prompt = "Valid examples of company name include: \nAECOM Asia Company Limited\nATAL Engineering Limited\nBased on the example provided and other common company name format, try to extract company name from following. No need to show any explanation. :\n"
    return llm.invoke(prompt + row['Contractor(s) & Address(es)'])

df3['ref']=df2['Tender Reference']
df3['description']=df2['Particulars']
df3['department'] = df2['department']
df3['awardee'] = df2.apply(get_gld_awardee, axis=1)  
df3['award_date']= df2.apply(get_gld_date, axis=1)  
df3['sum'] = df2.apply(get_gld_sum, axis=1)  
df3['url']=df2['url']
df3['batch_time'] = df2['batch_time']

df3.to_json('/Users/rowena/data/gld_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/gld_out.csv', index=False)   



In [ ]:
# GLD master (historical data)

# url = 'https://pcms2.gld.gov.hk/iprod/#/scn00101'
# browser.get (url)
# browser.implicitly_wait(60)
# soup = BeautifulSoup(browser.page_source, 'html.parser')
# batch_time = datetime.now() 


In [43]:
# HA 

url = 'https://www.ha.org.hk/visitor/ha_visitor_index.asp?Content_ID=2001&Lang=ENG&Dimension=10'
browser.get (url)
browser.implicitly_wait(60)
soup = BeautifulSoup(browser.page_source, 'html.parser')
batch_time = datetime.now() 

df = pd.DataFrame()
browser.switch_to.frame(browser.find_element ("name","childframe"))
url2 = browser.find_elements("xpath","//a[contains(text(),'CONTRACT AWARD NOTICE')]")[0].get_attribute('href')

browser2.get (url2)
browser2.implicitly_wait(120)
time.sleep(10)
#soup2 = BeautifulSoup(browser2.page_source, 'html.parser')
soup2 = BeautifulSoup(browser2.page_source, features='html')

t = soup2.select('table')[0] 
print(len(t.select('tr')))
for tr in t.select('tr'): 
    if re.search ('^Contract Award Notice$|^(Contract award)|Awarded Month:|^$', tr.text.strip()) :
        tr.decompose()

df = pd.DataFrame()
data = pd.read_html(io.StringIO(str(t)), header=0)
df = data[0]
df=df.rename(columns=df.iloc[0,:])
df = df.query(' not `Tender Reference`.str.contains("Tender Reference")', engine='python')

df['batch_time'] = batch_time
df['department'] = 'Hong Kong Hospital Authority'
df['url']= url2
df.to_json('/Users/rowena/data/ha_raw.json', orient = 'records', compression = 'infer', index = 'false')

df2 = pd.read_json('/Users/rowena/data/ha_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()
# ref, department, office, description, awardee, award_date, sum,  period, start, end url, batch_time

def get_ha_awardee (row): 
    prompt = "Valid examples of company name include: \nAECOM Asia Company Limited\nATAL Engineering Limited\nBased on the example provided and other common company name format, try to extract company name from following. No need to show any explanation. :\n"
    return llm.invoke(prompt + str(row['Contractor(s) & Address(es)']))


df3['ref']=df2['Tender Reference']
df3['description']=df2['Subject']
df3['department'] = df2['department']
df3['office'] = df2['Hospital']
df3['awardee'] = df2.apply(get_ha_awardee, axis=1)  
df3['award_date']= df2['Date of Award']
df3['sum'] = df2['Estimated  Contract Amount']
df3['period']= df2['Contract Period']
df3['url']=df2['url']
df3['batch_time'] = df2['batch_time']

df3.to_json('/Users/rowena/data/ha_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/ha_out.csv', index=False)   


1135


In [ ]:
#df.iloc[0,:]

df=df.rename(columns=df.iloc[0,:])

In [ ]:
df

In [45]:

df.to_json('/Users/rowena/data/ha_raw.json', orient = 'records', compression = 'infer', index = 'false')

df2 = pd.read_json('/Users/rowena/data/ha_raw.json', orient ='records', compression = 'infer')
df3 = pd.DataFrame()
# ref, department, office, description, awardee, award_date, sum,  period, start, end url, batch_time

def get_ha_awardee (row): 
    prompt = "Valid examples of company name include: \nAECOM Asia Company Limited\nATAL Engineering Limited\nBased on the example provided and other common company name format, try to extract company name from following. No need to show any explanation. :\n"
    return llm.invoke(prompt + str(row['Contractor(s) & Address(es)']))


df3['ref']=df2['Tender Reference']
df3['description']=df2['Subject']
df3['department'] = df2['department']
df3['office'] = df2['Hospital']
df3['awardee'] = df2.apply(get_ha_awardee, axis=1)  
df3['award_date']= df2['Date of Award']
df3['sum'] = df2['Estimated  Contract Amount']
df3['period']= df2['Contract Period']
df3['url']=df2['url']
df3['batch_time'] = df2['batch_time']

df3.to_json('/Users/rowena/data/ha_out.json', orient = 'records', compression = 'infer', index = 'false')
df3.to_csv('/Users/rowena/data/ha_out.csv', index=False)  

KeyboardInterrupt: 

In [47]:
df = pd.DataFrame()
df = df._append(pd.read_csv('/Users/rowena/data/epd_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/dsd_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/cesd_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/cesd2_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/emsd_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/emsd2_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/hyd_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/hyd2_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/wsd_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/wsd2_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/td_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/td2_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/hkaa_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/gld_out.csv'))
df = df._append(pd.read_csv('/Users/rowena/data/ha_out.csv'))
df.to_json('/Users/rowena/data/combined.json', orient = 'records', compression = 'infer', index = 'false')
df.to_csv('/Users/rowena/data/combined.csv', index=False)  


In [ ]:
os.rename('data', 'data_'+ datetime.now().strftime("%Y%m%d_%H%M%S"))